<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_08_Advanced_Machine_Learning/Week_2_Hyperparameter_Tuning/cross_validation_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Validation Strategies for Robust Model Evaluation
## Learning Objectives
- Understand bias-variance in model evaluation
- Implement k-fold, stratified, and time-series CV
- Use nested cross-validation for unbiased hyperparameter tuning


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut,
    cross_val_score, cross_validate, GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

X, y = load_breast_cancer(return_X_y=True)
print(f'Dataset: {X.shape}, Class balance: {np.bincount(y)}')

In [ ]:
# K-Fold vs Stratified K-Fold comparison
model = RandomForestClassifier(n_estimators=50, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kf_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
skf_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print(f'K-Fold scores:            {kf_scores.round(4)}')
print(f'K-Fold mean ± std:        {kf_scores.mean():.4f} ± {kf_scores.std():.4f}')
print(f'\nStratified K-Fold scores: {skf_scores.round(4)}')
print(f'Stratified mean ± std:    {skf_scores.mean():.4f} ± {skf_scores.std():.4f}')

In [ ]:
# Time series cross-validation
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt

tscv = TimeSeriesSplit(n_splits=5)

# Visualize splits on timeline
n_samples = 100
X_ts = np.arange(n_samples).reshape(-1, 1)
y_ts = np.sin(X_ts.ravel() / 10) + np.random.normal(0, 0.1, n_samples) > 0

print('Time Series CV splits (train/test sizes):')
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_ts), 1):
    print(f'  Fold {fold}: train={len(train_idx):3d} samples [{train_idx[0]}-{train_idx[-1]}], '
          f'test={len(test_idx):2d} samples [{test_idx[0]}-{test_idx[-1]}]')

In [ ]:
# Nested cross-validation for unbiased evaluation
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(random_state=42))])
param_grid = {'rf__n_estimators': [50, 100], 'rf__max_depth': [None, 5]}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

nested_scores = cross_val_score(
    GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='accuracy'),
    X, y, cv=outer_cv, scoring='accuracy'
)
print(f'Nested CV scores: {nested_scores.round(4)}')
print(f'Nested CV mean:   {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print('\nNested CV gives an unbiased estimate of generalization performance.')

## Exercises
1. Compare LOO, 5-Fold, and 10-Fold CV variance on a small dataset (n=50).
2. Implement walk-forward validation for a time series regression problem.
3. Use `cross_validate` to simultaneously compute multiple metrics.


## Summary
- Stratified K-Fold preserves class distribution in each fold.
- TimeSeriesSplit prevents data leakage in temporal data.
- Nested CV separates model selection from generalization estimation.
